# Ноутбук 3а — Подготовка датасета

**Пайплайн:** pseudo-labels → Label Studio → YOLO → augmentations



In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json, yaml, shutil, random
from pathlib import Path
from typing import List, Tuple, Dict
from tqdm import tqdm
from urllib.parse import quote

PAGES_DIR        = Path("test")                    # страницы из ноутбука 1
LABELSTUDIO_JSON = Path("labelstudio_export.json")   # экспорт из Label Studio
DATASET_DIR      = Path("data")
CLASSES = ["table", "cell", "merged_cell", "row"]   # порядок = class_id
VAL_RATIO    = 0.15
TEST_RATIO   = 0.15
RANDOM_SEED  = 42
AUG_COPIES   = 3   # аугм. копий на каждый train-файл
H_LINE_SCALE = 30
V_LINE_SCALE = 30
CELL_MIN_W   = 30
CELL_MIN_H   = 20

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print("Классы:", {i: c for i, c in enumerate(CLASSES)})

## Шаг 1 — Pseudo-labeling

CV-парсер генерирует черновые bbox, которые затем правятся в Label Studio.

> **Workflow:** запустить → импортировать JSON в Label Studio → поправить → экспортировать → Шаг 2

In [ ]:
def cv_detect(gray, h_scale=30, v_scale=30, min_w=30, min_h=20, max_w=4000, max_h=2000):
    h, w = gray.shape
    binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, 31, 10)
    hk = cv2.getStructuringElement(cv2.MORPH_RECT, (max(w//h_scale,5), 1))
    vk = cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(h//v_scale,5)))
    h_lines = cv2.dilate(cv2.morphologyEx(binary, cv2.MORPH_OPEN, hk),
                         cv2.getStructuringElement(cv2.MORPH_RECT,(1,2)))
    v_lines = cv2.dilate(cv2.morphologyEx(binary, cv2.MORPH_OPEN, vk),
                         cv2.getStructuringElement(cv2.MORPH_RECT,(2,1)))
    grid = cv2.add(h_lines, v_lines)
    conts, _ = cv2.findContours(cv2.bitwise_not(grid), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    boxes = [(x,y,bw,bh) for c in conts for x,y,bw,bh in [cv2.boundingRect(c)]
             if min_w<=bw<=max_w and min_h<=bh<=max_h]

    cells, merged, rows_bbox = [], [], []
    if boxes:
        med = np.median([bw*bh for _,_,bw,bh in boxes])
        for x,y,bw,bh in boxes:
            (merged if bw*bh > med*2.5 else cells).append([x,y,bw,bh])
        if cells:
            tol = np.median([b[3] for b in cells]) * 0.4
            groups = []
            for box in sorted(cells, key=lambda b: b[1]):
                placed = False
                for g in groups:
                    if abs(box[1]-g[0][1]) < tol:
                        g.append(box); placed = True; break
                if not placed: groups.append([box])
            for g in groups:
                rx,ry = min(b[0] for b in g), min(b[1] for b in g)
                rows_bbox.append([rx,ry,
                    max(b[0]+b[2] for b in g)-rx, max(b[1]+b[3] for b in g)-ry])

    table = []
    if boxes:
        tx,ty = min(b[0] for b in boxes), min(b[1] for b in boxes)
        table = [[tx,ty,max(b[0]+b[2] for b in boxes)-tx,max(b[1]+b[3] for b in boxes)-ty]]
    return {"table":table,"cell":cells,"merged_cell":merged,"row":rows_bbox}


page_files = sorted(PAGES_DIR.glob("*.jpg")) + sorted(PAGES_DIR.glob("*.png"))
print(f"Страниц для pseudo-labeling: {len(page_files)}")

ls_tasks = []
for img_path in tqdm(page_files, desc='Обработка', unit='файл', colour='green'):
    abs_path = img_path.resolve().as_posix()
    img = cv2.imread(str(img_path))
    if img is None: continue
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ih, iw = gray.shape
    dets = cv_detect(gray, H_LINE_SCALE, V_LINE_SCALE, CELL_MIN_W, CELL_MIN_H)
    results = []
    for cls_name, bxs in dets.items():
        for x,y,bw,bh in bxs:
            results.append({"type":"rectanglelabels",
                "value":{"x":x/iw*100,"y":y/ih*100,
                         "width":bw/iw*100,"height":bh/ih*100,
                         "rectanglelabels":[cls_name]},
                "score":0.5,"from_name":"label",
                "to_name":"image","origin":"prediction"})
    ls_tasks.append({'data': {'image': f'file://{abs_path}'},
                     "predictions":[{"result":results,"score":0.5}]})

out = Path("pseudo_labels_for_labelstudio.json")
out.write_text(json.dumps(ls_tasks, ensure_ascii=False, indent=2))
print(f"Сохранено → {out}")
print("Следующий шаг: импортируй в Label Studio, поправь, экспортируй обратно в JSON")

In [ ]:
!label-studio

## Шаг 2 — Label Studio → YOLO

Экспортируй из Label Studio в формате **JSON (Label Studio native)**.
Формат YOLO:  (нормировано 0–1).

In [ ]:
def ls_to_yolo(ls_json, images_src, out_dir, classes):
    out_dir.mkdir(parents=True, exist_ok=True)
    cls2id = {c: i for i, c in enumerate(classes)}
    tasks = json.loads(ls_json.read_text(encoding="utf-8"))
    img_paths, skipped = [], 0
    for task in tasks:
        img_name = Path(task.get("data",{}).get("image","")).name
        cands = list(images_src.glob(f"**/{img_name}"))
        if not cands: skipped+=1; continue
        img = cv2.imread(str(cands[0]))
        if img is None: skipped+=1; continue
        anns = task.get("annotations",[])
        if not anns: skipped+=1; continue
        lines = []
        for r in anns[0].get("result",[]):
            if r.get("type") != "rectanglelabels": continue
            v = r["value"]
            label = (v.get("rectanglelabels") or [""])[0]
            if label not in cls2id: continue
            xp,yp,wp,hp = v["x"]/100,v["y"]/100,v["width"]/100,v["height"]/100
            lines.append(f"{cls2id[label]} {xp+wp/2:.6f} {yp+hp/2:.6f} {wp:.6f} {hp:.6f}")
        if not lines: skipped+=1; continue
        src = cands[0]
        shutil.copy2(src, out_dir/src.name)
        (out_dir/f"{src.stem}.txt").write_text("".join(lines))
        img_paths.append(out_dir/src.name)
    print(f"Конвертировано: {len(img_paths)}  |  пропущено: {skipped}")
    return img_paths


raw_dir = DATASET_DIR / "raw"
img_paths = ls_to_yolo(LABELSTUDIO_JSON, PAGES_DIR, raw_dir, CLASSES)
print(f"Всего: {len(img_paths)} страниц")

## Шаг 3 — Разбивка train/val/test

In [ ]:
def split_dataset(img_paths, ddir, val_r, test_r, seed):
    rng = random.Random(seed)
    paths = list(img_paths); rng.shuffle(paths)
    n = len(paths); nt = max(1,int(n*test_r)); nv = max(1,int(n*val_r))
    splits = {"test":paths[:nt],"val":paths[nt:nt+nv],"train":paths[nt+nv:]}
    for split, sp in splits.items():
        idir = ddir/split/"images"; idir.mkdir(parents=True, exist_ok=True)
        ldir = ddir/split/"labels"; ldir.mkdir(parents=True, exist_ok=True)
        for src in sp:
            shutil.copy2(src, idir/src.name)
            lbl = src.with_suffix(".txt")
            if lbl.exists(): shutil.copy2(lbl, ldir/lbl.name)
        print(f"  {split:5s}: {len(sp)} изображений")
    return splits


print("Разбивка датасета:")
splits = split_dataset(img_paths, DATASET_DIR, VAL_RATIO, TEST_RATIO, RANDOM_SEED)

yaml.dump({"path":str(DATASET_DIR.resolve()),
           "train":"train/images","val":"val/images","test":"test/images",
           "nc":len(CLASSES),"names":CLASSES},
          open(DATASET_DIR/"data.yaml","w"), allow_unicode=True, sort_keys=False)
print(f"data.yaml → {DATASET_DIR}/data.yaml")

## Шаг 4 — Аугментации под архивные документы

- **Наклон ±3°** — документ плохо лежал на планшете
- **Затемнение угла** — неравномерное освещение сканера
- **Гауссовый шум** — старая зернистая бумага
- **Размытие** — нечёткий скан
- **Яркость/контраст** — выцветшие чернила

In [ ]:
def read_yolo(path):
    if not path.exists() or path.stat().st_size==0: return np.empty((0,5))
    return np.loadtxt(path, dtype=np.float32).reshape(-1,5)

def write_yolo(path, labels):
    if not len(labels): return
    path.write_text("".join(
        f"{int(r[0])} {r[1]:.6f} {r[2]:.6f} {r[3]:.6f} {r[4]:.6f}" for r in labels))

def rotate_labels(labels, angle_deg):
    if not len(labels): return labels
    rad = np.radians(-angle_deg); c,s = np.cos(rad),np.sin(rad)
    out = labels.copy()
    dx,dy = labels[:,1]-0.5, labels[:,2]-0.5
    out[:,1] = np.clip(0.5+dx*c-dy*s, 0, 1)
    out[:,2] = np.clip(0.5+dx*s+dy*c, 0, 1)
    return out

def augment(img, labels, rng):
    h,w = img.shape[:2]; out = img.copy()
    # Наклон ±3°
    angle = rng.uniform(-3,3)
    M = cv2.getRotationMatrix2D((w//2,h//2), angle, 1.0)
    out = cv2.warpAffine(out, M, (w,h), borderMode=cv2.BORDER_REPLICATE)
    labels = rotate_labels(labels, angle)
    # Затемнение угла
    if rng.random() < 0.6:
        corner = rng.integers(0,4)
        grad = int(min(h,w)*rng.uniform(0.25,0.6))
        alpha = rng.uniform(0.4, 0.85)
        mask = np.ones((h,w), np.float32)
        for i in range(grad):
            v = alpha+(1-alpha)*(i/grad); s = grad-i
            sl = [(slice(None,s),slice(None,s)),(slice(None,s),slice(w-s,None)),
                  (slice(h-s,None),slice(None,s)),(slice(h-s,None),slice(w-s,None))][corner]
            mask[sl] = np.minimum(mask[sl], v)
        out = np.clip(out*mask[:,:,None], 0, 255).astype(np.uint8)
    # Гауссовый шум
    if rng.random() < 0.5:
        noise = rng.normal(0, rng.uniform(3,12), out.shape).astype(np.int16)
        out = np.clip(out.astype(np.int16)+noise, 0, 255).astype(np.uint8)
    # Размытие
    if rng.random() < 0.4:
        k = int(rng.choice([3,5]))
        out = cv2.GaussianBlur(out, (k,k), 0)
    # Яркость/контраст
    if rng.random() < 0.5:
        out = np.clip(out.astype(np.float32)*rng.uniform(0.85,1.15)
                      +rng.uniform(-15,15), 0, 255).astype(np.uint8)
    return out, labels


train_idir = DATASET_DIR/"train"/"images"
train_ldir = DATASET_DIR/"train"/"labels"
rng = np.random.default_rng(RANDOM_SEED)
aug_count = 0

for ip in sorted(train_idir.glob("*.jpg")):
    img = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)
    lbls = read_yolo(train_ldir/(ip.stem+".txt"))
    for k in range(AUG_COPIES):
        aug_img, aug_lbl = augment(img, lbls, rng)
        stem = f"{ip.stem}_aug{k:02d}"
        cv2.imwrite(str(train_idir/f"{stem}.jpg"),
                    cv2.cvtColor(aug_img,cv2.COLOR_RGB2BGR),[cv2.IMWRITE_JPEG_QUALITY,90])
        write_yolo(train_ldir/f"{stem}.txt", aug_lbl)
        aug_count += 1

total = len(list(train_idir.glob("*.jpg")))
print(f"Аугментаций: {aug_count}  |  Итого train: {total}")

## Шаг 5 — Визуальная проверка датасета

Меняй  чтобы проверять разные части.

In [ ]:
PREVIEW_SPLIT = "train"   # "train" | "val" | "test"
N_PREVIEW     = 4
COLORS = {0:(255,80,80), 1:(80,160,255), 2:(80,220,120), 3:(255,180,50)}

idir = DATASET_DIR/PREVIEW_SPLIT/"images"
ldir = DATASET_DIR/PREVIEW_SPLIT/"labels"
files = sorted(idir.glob("*.jpg"))
chosen = random.sample(files, min(N_PREVIEW, len(files)))

fig, axes = plt.subplots(1, len(chosen), figsize=(6*len(chosen), 10))
if len(chosen) == 1: axes = [axes]
for ax, ip in zip(axes, chosen):
    img = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)
    ih, iw = img.shape[:2]
    for cls,cx,cy,bw,bh in read_yolo(ldir/(ip.stem+".txt")):
        x1,y1 = int((cx-bw/2)*iw), int((cy-bh/2)*ih)
        x2,y2 = int((cx+bw/2)*iw), int((cy+bh/2)*ih)
        c = COLORS.get(int(cls),(200,200,200))
        cv2.rectangle(img,(x1,y1),(x2,y2),c,2)
        cv2.putText(img,CLASSES[int(cls)],(x1+2,y1+14),cv2.FONT_HERSHEY_SIMPLEX,0.4,c,1)
    ax.imshow(img); ax.set_title(ip.name, fontsize=9); ax.axis("off")

patches = [mpatches.Patch(color=np.array(c)/255,label=CLASSES[i]) for i,c in COLORS.items()]
fig.legend(handles=patches, loc="lower center", ncol=4, fontsize=10)
plt.suptitle(f"Проверка ({PREVIEW_SPLIT})", fontsize=13)
plt.tight_layout(); plt.show()

all_lbls = np.vstack([read_yolo(ldir/(p.stem+".txt")) for p in files
                      if (ldir/(p.stem+".txt")).exists()]) if files else np.empty((0,5))
print(f"Статистика {PREVIEW_SPLIT}:")
for i,name in enumerate(CLASSES):
    cnt = int((all_lbls[:,0]==i).sum()) if len(all_lbls) else 0
    print(f"  {name:15s}: {cnt:5d} bbox")